<a href="https://colab.research.google.com/github/karanveer-sharma/pythontraining_with_ai/blob/main/captionproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
np.random.seed(42)

In [ ]:
print("data analysis:")
print("\n[STEP 1] ")
servers = [f"SRV-{i:03d}" for i in range(101, 121)] * 5
dates = pd.date_range("2024-01-01", periods=100)

logs_dict = {
    "LogID": range(1, 101),
    "Date": dates,
    "ServerID": servers,
    "RAM": np.random.choice(["8GB", "16GB", "4096MB", "8192MB", "32GB", np.nan], 100),
    "CPU": np.random.choice(["45%", "90%", "12%", "105%", "5%", np.nan], 100),
    "Region": np.random.choice(["North-East", "north-east", "West", "South", "west"], 100)
}

data analysis:

[STEP 1] 


In [ ]:
df_logs = pd.DataFrame(logs_dict)
df_logs = pd.concat([df_logs, df_logs.iloc[:5]], ignore_index=True)

tickets_dict = {
    "TicketID": [f"TKT-{i}" for i in range(1001, 1081)],
    "ServerRef": np.random.choice(servers[:15], 80),
    "Severity": np.random.choice(["High", "Medium", "Low", "CRITICAL", "Critical"], 80),
    "ResolutionHrs": np.random.uniform(1, 48, 80)
}

df_tickets = pd.DataFrame(tickets_dict)
df_tickets.loc[79, "ServerRef"] = "SRV-999"

print(f"\nLogs created: {len(df_logs)} rows")
print(df_logs.head(3))
print(f"\nTickets created: {len(df_tickets)} rows")
print(df_tickets.head(3))



Logs created: 105 rows
   LogID       Date ServerID     RAM  CPU      Region
0      1 2024-01-01  SRV-101  8192MB   5%  North-East
1      2 2024-01-02  SRV-102    32GB  45%       South
2      3 2024-01-03  SRV-103  4096MB  45%  North-East

Tickets created: 80 rows
   TicketID ServerRef Severity  ResolutionHrs
0  TKT-1001   SRV-114   Medium      12.643357
1  TKT-1002   SRV-105      Low      17.730716
2  TKT-1003   SRV-104   Medium      36.618767


In [ ]:
print("\nSTEP 2")
before = len(df_logs)
df_logs = df_logs.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(df_logs)} duplicates. Remaining: {len(df_logs)} rows")
df_logs["Region"] = df_logs["Region"].str.title()
print("Regions standardized:", df_logs["Region"].unique())

def ram_to_gb(val):
    if pd.isna(val): return np.nan
    s = str(val).upper()
    if "GB" in s: return int(s.replace("GB", ""))
    if "MB" in s: return int(s.replace("MB", "")) / 1024
    return np.nan

df_logs["RAM"] = df_logs["RAM"].apply(ram_to_gb)
df_logs["RAM"] = df_logs["RAM"].fillna(df_logs["RAM"].median()).astype(int)

df_logs["CPU"] = df_logs["CPU"].str.replace("%", "", regex=False).astype(float)
df_logs["CPU"] = df_logs["CPU"].apply(lambda x: min(x, 100) if pd.notna(x) else x)

print(df_logs.head(3))



STEP 2
Removed 5 duplicates. Remaining: 100 rows
Regions standardized: ['North-East' 'South' 'West']
   LogID       Date ServerID  RAM   CPU      Region
0      1 2024-01-01  SRV-101    8   5.0  North-East
1      2 2024-01-02  SRV-102   32  45.0       South
2      3 2024-01-03  SRV-103    4  45.0  North-East


In [ ]:
print("\nSTEP 3")

df_tickets["Severity"] = df_tickets["Severity"].str.strip().str.capitalize()
print("Unique severities:", df_tickets["Severity"].unique())



STEP 3
Unique severities: ['Medium' 'Low' 'High' 'Critical']


In [ ]:
print("\nSTEP 4")

df_merged = df_logs.merge(df_tickets, left_on="ServerID", right_on="ServerRef", how="left")
print("Merged rows:", len(df_merged))

ghosts = df_tickets[~df_tickets["ServerRef"].isin(df_logs["ServerID"])]
if not ghosts.empty:
    print("Ghost servers detected:", ghosts["ServerRef"].unique())



STEP 4
Merged rows: 420
Ghost servers detected: ['SRV-999']


In [ ]:
print("\nSTEP 5")

def status(row):
    if pd.notna(row["CPU"]) and pd.notna(row["Severity"]):
        if row["CPU"] > 80 and row["Severity"] == "Critical":
            return "DANGER"
    return "Normal"

df_merged["Status"] = df_merged.apply(status, axis=1)
print(df_merged["Status"].value_counts())



STEP 5
Status
Normal    378
DANGER     42
Name: count, dtype: int64


In [ ]:
print("\nSTEP 6")

report = df_merged.groupby("ServerID").agg({
    "CPU": "mean",
    "TicketID": "count",
    "ResolutionHrs": "mean"
}).reset_index()

report.columns = ["ServerID", "AvgCPU", "Tickets", "AvgResolution"]
report["AvgCPU"] = report["AvgCPU"].round(2)
report["AvgResolution"] = report["AvgResolution"].round(2)

print(report)



STEP 6
   ServerID  AvgCPU  Tickets  AvgResolution
0   SRV-101   64.00       45          28.96
1   SRV-102   63.00       15          21.65
2   SRV-103   40.50       45          30.37
3   SRV-104   18.33       15          31.56
4   SRV-105   31.80       35          21.83
5   SRV-106   70.00       35          17.87
6   SRV-107   39.00       15          16.60
7   SRV-108   34.80       30          23.15
8   SRV-109   53.80       10          30.22
9   SRV-110   52.33       10          39.86
10  SRV-111   28.20       15          15.40
11  SRV-112   40.00       35          16.06
12  SRV-113   12.00       25          20.97
13  SRV-114    6.75       40          19.73
14  SRV-115   58.40       25          30.47
15  SRV-116   40.40        0            NaN
16  SRV-117   67.40        0            NaN
17  SRV-118   64.25        0            NaN
18  SRV-119   23.00        0            NaN
19  SRV-120   72.50        0            NaN


In [ ]:
print("\nSTEP 7")

top5 = report.nlargest(5, "AvgCPU")
for _, r in top5.iterrows():
    res_time = f"{r['AvgResolution']}h" if r["Tickets"] > 0 else "N/A"
    print(f"{r['ServerID']} | CPU {r['AvgCPU']}% | Tickets {r['Tickets']} | Resolution {res_time}")

danger_servers = df_merged[df_merged["Status"] == "DANGER"]["ServerID"].unique()
print("\nCritical servers:", danger_servers)


STEP 7
SRV-120 | CPU 72.5% | Tickets 0 | Resolution N/A
SRV-106 | CPU 70.0% | Tickets 35 | Resolution 17.87h
SRV-117 | CPU 67.4% | Tickets 0 | Resolution N/A
SRV-118 | CPU 64.25% | Tickets 0 | Resolution N/A
SRV-101 | CPU 64.0% | Tickets 45 | Resolution 28.96h

Critical servers: ['SRV-106' 'SRV-115' 'SRV-101' 'SRV-102' 'SRV-103' 'SRV-108' 'SRV-111'
 'SRV-112']
